1) STAGING TABLES

Purpose:
- Read Silver tables as streaming sources
- Drop Silver/Bronze audit columns
- Add Gold audit columns:
gold_loaded_at, gold_updated_at
- Add lineage column:
source_table (silver table name)
- Add bigint surrogate keys where needed

In [0]:
-- STG: USERS

CREATE OR REFRESH STREAMING TABLE coffee.gold.stg_users (
  CONSTRAINT valid_user_id EXPECT (user_id IS NOT NULL),
  CONSTRAINT valid_registered_at EXPECT (registered_at IS NOT NULL)
)
COMMENT "Staging for users from silver: selects business columns + adds gold audit + lineage"
AS
SELECT
  -- Business key
  user_id,

  -- Descriptive attributes
  gender,
  birthdate,
  registered_at,

  -- Required for SCD sequencing
  silver_updated_at,

  -- Gold lineage column
  'coffee.silver.users' AS source_table,

  -- Gold audit timestamps
  current_timestamp() AS gold_loaded_at,
  current_timestamp() AS gold_updated_at
FROM STREAM(coffee.silver.users);


In [0]:
-- STG: STORES

CREATE OR REFRESH STREAMING TABLE coffee.gold.stg_stores
COMMENT "Staging for stores from silver: selects business columns + adds gold audit + lineage"
AS
SELECT
  -- Business key
  store_id,

  -- Descriptive attributes
  store_name,
  street,
  city,
  state,
  postal_code,
  latitude,
  longitude,

  -- Required for SCD sequencing
  silver_updated_at,

  -- Gold lineage column
  'coffee.silver.stores' AS source_table,

  -- Gold audit timestamps
  current_timestamp() AS gold_loaded_at,
  current_timestamp() AS gold_updated_at
FROM STREAM(coffee.silver.stores);


In [0]:
-- STG: MENU ITEMS

CREATE OR REFRESH STREAMING TABLE coffee.gold.stg_menu_items
COMMENT "Staging for menu items from silver: selects business columns + gold audit + lineage"
AS
SELECT
  -- Business key
  item_id,

  -- Descriptive attributes
  item_name,
  category,
  price,
  is_seasonal,
  available_from,
  available_to,

  -- Required for SCD sequencing
  silver_updated_at,

  -- Gold lineage column
  'coffee.silver.menu_items' AS source_table,

  -- Gold audit timestamps
  current_timestamp() AS gold_loaded_at,
  current_timestamp() AS gold_updated_at
FROM STREAM(coffee.silver.menu_items);


In [0]:
-- STG: VOUCHERS

CREATE OR REFRESH STREAMING TABLE coffee.gold.stg_vouchers
COMMENT "Staging for vouchers from silver: selects business columns + adds gold audit + lineage"
AS
SELECT
  -- Business key
  voucher_id,

  -- Descriptive attributes
  voucher_code,
  discount_type,
  discount_value,
  valid_from,
  valid_to,

  -- Required for SCD sequencing
  silver_updated_at,

  -- Gold lineage column
  'coffee.silver.vouchers' AS source_table,

  -- Gold audit timestamps
  current_timestamp() AS gold_loaded_at,
  current_timestamp() AS gold_updated_at
FROM STREAM(coffee.silver.vouchers);


In [0]:
-- STG: PAYMENT METHODS

CREATE OR REFRESH STREAMING TABLE coffee.gold.stg_payment_methods
COMMENT "Staging for payment methods from silver: selects business columns + adds gold audit + lineage"
AS
SELECT
  -- Business key
  method_id,

  -- Descriptive attributes
  method_name,
  category,

  -- Required for SCD sequencing
  silver_updated_at,

  -- Gold lineage column
  'coffee.silver.payment_methods' AS source_table,

  -- Gold audit timestamps
  current_timestamp() AS gold_loaded_at,
  current_timestamp() AS gold_updated_at
FROM STREAM(coffee.silver.payment_methods);


In [0]:
-- STG: TRANSACTIONS

CREATE OR REFRESH STREAMING TABLE coffee.gold.stg_transactions (
  CONSTRAINT valid_transaction_id EXPECT (transaction_id IS NOT NULL),
  CONSTRAINT valid_created_at EXPECT (created_at IS NOT NULL),
  CONSTRAINT valid_final_amount EXPECT (final_amount >= 0)
)
COMMENT "Gold staging for transactions: business columns + silver_updated_at + gold audit + lineage"
AS
SELECT
  transaction_id,
  store_id,
  payment_method_id,
  voucher_id,
  user_id,
  original_amount,
  discount_applied,
  final_amount,
  created_at,

  -- Silver audit column used for sequencing
  silver_updated_at,

  -- Lineage
  'coffee.silver.transactions' AS source_table,

  -- Gold audit
  current_timestamp() AS gold_loaded_at,
  current_timestamp() AS gold_updated_at
FROM STREAM(coffee.silver.transactions);


In [0]:
-- ==========================================================
-- STG: TRANSACTION ITEMS (ENRICHED)
--
-- Adds store_id by joining with stg_transactions.
-- This is required for Unity Catalog Row Level Security (RLS)
-- because nested row filters (items -> transactions) are not supported.
-- ==========================================================

CREATE OR REFRESH STREAMING TABLE coffee.gold.stg_transaction_items
COMMENT "Gold staging for transaction items: adds store_id + bigint key + silver_updated_at + gold audit + lineage"
AS
SELECT
  i.transaction_item_sk,
  i.transaction_id,
  t.store_id,                     --  Added for RLS
  i.item_id,
  i.quantity,
  i.unit_price,
  i.subtotal,
  i.created_at,

  -- Silver audit column used for sequencing
  i.silver_updated_at,

  -- Bigint surrogate key for evaluator/BI tools
  xxhash64(i.transaction_item_sk) AS transaction_item_key,

  -- Lineage
  'coffee.silver.transaction_items' AS source_table,

  -- Gold audit
  current_timestamp() AS gold_loaded_at,
  current_timestamp() AS gold_updated_at

FROM STREAM(coffee.silver.transaction_items) i
LEFT JOIN coffee.silver.transactions t

  ON i.transaction_id = t.transaction_id;
